# Complexity-Debt Trajectories — FINAL
### Forecasting complexity accumulation in Python repositories

End-to-end on Colab: mine → landmark features (incl. structure & coupling) → survival models → **paper-ready tables**.

**Established across five iterations on real data (14 repos, 36,076 file-commits):**

| Model | 5×2 CV concordance |
|---|---|
| Static (McCabe) | 0.539 |
| Static (Halstead) | 0.529 |
| Process (n_authors) | 0.563 |
| **Cox (pruned)** | **~0.66** |

**The three honest findings this notebook is built to report:**
1. **Main claim (supported):** the trajectory model beats every prior-work baseline under repeated CV; paired gain +0.085 [+0.035, +0.152], significant in 8/9 landmark configurations.
2. **Structural covariates do NOT add predictive power** over process covariates (contrast +0.014 [−0.026, +0.050]) — nor does a nonlinear model (+0.021 [−0.017, +0.078]). Reported as a negative result, not hidden.
3. **Generalization is weak.** Leave-one-repo-out drops the model to 0.599 (McCabe 0.588); it loses on 4/14 unseen projects. The paper must state this plainly.

**Fixes in this version:**
- **Collinearity removed.** `fan_out`↔`n_imports` correlated 0.79 and split one signal into opposite coefficients — an artifact, not a finding. Pruned to 14 covariates; max VIF 8.6→2.9. **Coefficient signs are no longer interpreted.**
- **Leave-one-repo-out** added as the primary generalization test.
- **Paper tables** (Table 2 results, Table 3 LORO, Table 4 sensitivity) exported as CSV.

In [ ]:
!pip install -q pydriller radon
print("ok")

In [ ]:
# 2) config
import warnings; warnings.filterwarnings("ignore")
import ast, os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import defaultdict
from scipy.optimize import minimize
from numpy.linalg import lstsq

REPOS = [
    "https://github.com/psf/requests","https://github.com/pallets/flask",
    "https://github.com/pallets/click","https://github.com/encode/httpx",
    "https://github.com/scrapy/scrapy","https://github.com/Textualize/rich",
    "https://github.com/tiangolo/fastapi","https://github.com/celery/celery",
    "https://github.com/getsentry/sentry-python","https://github.com/sqlalchemy/sqlalchemy",
    "https://github.com/sphinx-doc/sphinx","https://github.com/pypa/pipenv",
    "https://github.com/streamlit/streamlit","https://github.com/apache/airflow",
]
MAX_COMMITS_PER_REPO = 2500   # keeps EARLIEST commits -> truncates late life -> inflates censoring (report it)
FILE_EXT="_.py".replace("_","")  # ".py"
LANDMARK_L=3; DELTA=2.0; PENALIZER=0.5; SEED=42
rng=np.random.default_rng(SEED)

# 14-covariate PRUNED set (collinear members removed: n_imports, hv, cc_density, nfunc, nesting_slope,
# n_classes, n_returns). max VIF 2.9.
BASE=["cc_lm","cc_slope","loc_lm","loc_delta","mi_lm","n_authors","lm_days"]
STRUCT=["nesting_lm","comment_ratio_lm","docstring_ratio_lm","max_params_lm","n_except_lm",
        "fan_in_lm","fan_out_lm"]
COV=BASE+STRUCT
print(f"{len(REPOS)} repos | {len(COV)} covariates | event: max_cc >= init+{DELTA} | landmark={LANDMARK_L}")

In [ ]:
# 3) extractors (AST + time-ordered import graph). All unit-tested before shipping.
def ast_features(src):
    f=dict(nesting_lm=np.nan,comment_ratio_lm=np.nan,docstring_ratio_lm=np.nan,
           max_params_lm=np.nan,n_except_lm=np.nan)
    if not src: return f
    try: tree=ast.parse(src)
    except Exception: return f
    def depth(n,d=0):
        b=d
        for ch in ast.iter_child_nodes(n):
            nd=d+isinstance(ch,(ast.If,ast.For,ast.While,ast.With,ast.Try,
                                ast.FunctionDef,ast.AsyncFunctionDef))
            b=max(b,depth(ch,nd))
        return b
    funcs=[n for n in ast.walk(tree) if isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef))]
    classes=[n for n in ast.walk(tree) if isinstance(n,ast.ClassDef)]
    f["nesting_lm"]=depth(tree)
    f["max_params_lm"]=max([len(fn.args.args) for fn in funcs],default=0)
    f["n_except_lm"]=len([n for n in ast.walk(tree) if isinstance(n,ast.ExceptHandler)])
    lines=[l for l in src.split("\n") if l.strip()]
    f["comment_ratio_lm"]=len([l for l in lines if l.strip().startswith("#")])/max(len(lines),1)
    doc=sum(1 for n in funcs+classes+[tree] if ast.get_docstring(n))
    f["docstring_ratio_lm"]=doc/max(len(funcs)+len(classes)+1,1)
    return f

def imported_modules(src):
    out=set()
    if not src: return out
    try: tree=ast.parse(src)
    except Exception: return out
    for n in ast.walk(tree):
        if isinstance(n,ast.Import):
            for a in n.names: out.add(a.name.split(".")[0])
        elif isinstance(n,ast.ImportFrom):
            if n.module: out.add(n.module.split(".")[0])
    return out

def module_name(path):
    parts=path.replace("\\","/").split("/"); fn=parts[-1]
    if fn=="__init__.py": return parts[-2] if len(parts)>1 else None
    return fn[:-3] if fn.endswith(".py") else None

class ImportGraph:
    def __init__(s): s.file_imports={}; s.importers=defaultdict(set)
    def update(s,path,imp):
        old=s.file_imports.get(path,set())
        for m in old-imp: s.importers[m].discard(path)
        for m in imp-old: s.importers[m].add(path)
        s.file_imports[path]=imp
    def fan_in(s,path):
        m=module_name(path); return 0 if m is None else len(s.importers.get(m,set())-{path})
    def fan_out(s,path): return len(s.file_imports.get(path,set()))

def radon_metrics(src):
    from radon.complexity import cc_visit
    from radon.metrics import mi_visit
    loc=tcc=mcc=nf=mi=np.nan
    if not src: return loc,tcc,mcc,nf,mi
    try: loc=src.count("\n")+1
    except Exception: pass
    try:
        b=cc_visit(src)
        if b:
            cs=[x.complexity for x in b]; tcc,mcc,nf=float(sum(cs)),float(max(cs)),len(cs)
    except Exception: pass
    try: mi=float(mi_visit(src,multi=True))
    except Exception: pass
    return loc,tcc,mcc,nf,mi
print("extractors ready")

In [ ]:
# 4) mine
def mine(repos):
    from pydriller import Repository
    rows=[]; failed=[]; skipped=0
    for repo in repos:
        name=repo.rstrip("/").split("/")[-1].replace(".git","")
        print(f"[+] {name}",flush=True); n=0; G=ImportGraph()
        try:
            for c in Repository(repo).traverse_commits():
                if MAX_COMMITS_PER_REPO and n>=MAX_COMMITS_PER_REPO: break
                for mf in c.modified_files:
                    p=mf.new_path or mf.old_path
                    if not p or not p.endswith(FILE_EXT): continue
                    s=mf.source_code
                    if s is None: continue
                    G.update(p,imported_modules(s))
                    loc,tcc,mcc,nf,mi=radon_metrics(s)
                    if np.isnan(mcc): skipped+=1
                    rows.append(dict(repo=name,commit_date=c.committer_date.isoformat(),
                        author_email=c.author.email,file_path=p,loc=loc,max_cc=mcc,mi=mi,
                        fan_in=G.fan_in(p),fan_out=G.fan_out(p),**ast_features(s)))
                n+=1
                if n%500==0: print(f"    {name}: {n}",flush=True)
        except Exception as ex: failed.append(name); print(f"[!] {name} FAILED: {ex}")
        print(f"[+] {name}: {n}",flush=True)
    print(f"\n[i] unparseable snapshots skipped: {skipped}")
    if failed: print("[!] failed repos (REPORT):",failed)
    return pd.DataFrame(rows)

df=mine(REPOS)
df["commit_date"]=pd.to_datetime(df["commit_date"],utc=True,errors="coerce")
df=df.dropna(subset=["commit_date","max_cc"]).sort_values(["repo","file_path","commit_date"])
df.to_csv("complexity_long_final.csv",index=False)
print("rows:",len(df),"| files:",df.file_path.nunique(),"| repos:",df.repo.nunique())

In [ ]:
# 5) landmark table + guards
def build_landmark(df,L=LANDMARK_L,delta=DELTA):
    recs=[]; short=early=0
    for (r,p),g in df.groupby(["repo","file_path"],sort=False):
        g=g.reset_index(drop=True)
        if len(g)<L+2: short+=1; continue
        init=g["max_cc"].iloc[0]; target=init+delta; lm=g.iloc[:L]
        if (lm["max_cc"]>=target).any(): early+=1; continue
        t_lm=lm["commit_date"].iloc[-1]; post=g.iloc[L:]
        cr=np.where(post["max_cc"].values>=target)[0]; ev=1 if cr.size else 0
        t_end=post["commit_date"].iloc[int(cr[0])] if cr.size else post["commit_date"].iloc[-1]
        a,b=lm.iloc[-1],lm.iloc[0]
        recs.append(dict(repo=r,file_path=p,duration_days=max((t_end-t_lm).days,1),event=ev,init_cc=init,
            cc_lm=a["max_cc"],cc_slope=a["max_cc"]-b["max_cc"],loc_lm=a["loc"],
            loc_delta=a["loc"]-b["loc"],mi_lm=a["mi"],n_authors=lm["author_email"].nunique(),
            lm_days=max((t_lm-b["commit_date"]).days,1),nesting_lm=a["nesting_lm"],
            comment_ratio_lm=a["comment_ratio_lm"],docstring_ratio_lm=a["docstring_ratio_lm"],
            max_params_lm=a["max_params_lm"],n_except_lm=a["n_except_lm"],
            fan_in_lm=a["fan_in"],fan_out_lm=a["fan_out"]))
    print(f"[i] excluded {short} too-short, {early} evented-in-landmark")
    return pd.DataFrame(recs).dropna()

S=build_landmark(df)
print("files:",len(S),"| events:",int(S.event.sum()),"| rate:",round(S.event.mean(),3))
tau=abs(S.init_cc.corr(S.event)); print(f"[guard] tautology |corr(init_cc,event)|={tau:.3f}")
assert tau<0.20
circ=[c for c in COV if abs(S[c].corr(S.duration_days))>0.5]; assert not circ, f"circular:{circ}"
def maxvif(S,cov):
    X=S[cov].values;X=(X-X.mean(0))/(X.std(0)+1e-9);m=0
    for i in range(len(cov)):
        o=np.delete(X,i,1);bb=lstsq(o,X[:,i],rcond=None)[0]
        r2=1-((X[:,i]-o@bb)**2).sum()/((X[:,i]-X[:,i].mean())**2).sum();m=max(m,1/max(1-r2,1e-9))
    return m
print(f"[guard] max VIF = {maxvif(S,COV):.1f} (want <5)")
S.to_csv("survival_landmark_final.csv",index=False); S.head()

In [ ]:
# 6) model + metrics
def concordance(t,risk,e):
    t=np.asarray(t,float);risk=np.asarray(risk,float);e=np.asarray(e,int);num=den=0.
    for i in np.where(e==1)[0]:
        m=t>t[i];den+=m.sum();num+=(risk[i]>risk[m]).sum()+.5*(risk[i]==risk[m]).sum()
    return num/den if den else np.nan
def fit_cox(X,t,e,pen=PENALIZER):
    o=np.argsort(t);X,t,e=X[o],t[o],e[o];ev=np.where(e==1)[0]
    def nll(b):
        r=X@b;ll=0.
        for i in ev: ll+=r[i]-np.log(np.exp(r[t>=t[i]]).sum())
        return -ll+pen*(b**2).sum()
    return minimize(nll,np.zeros(X.shape[1]),method="L-BFGS-B").x
def oriented(t,x,e): return max(concordance(t,x,e),concordance(t,-x,e))
print("model ready")

In [ ]:
# 7) TABLE 2 -- repeated CV, ours vs baselines, paired CI
t=S.duration_days.values.astype(float); e=S.event.values.astype(int)
X=S[COV].values; mu,sd=X.mean(0),X.std(0)+1e-9
def repeated_cv(reps=10,folds=5,seed=0):
    r=np.random.default_rng(seed); out=defaultdict(list); dif=[]
    for _ in range(reps):
        idx=r.permutation(len(S))
        for f in np.array_split(idx,folds):
            tr=np.setdiff1d(idx,f)
            if e[tr].sum()<5 or e[f].sum()<2: continue
            o=concordance(t[f],((X[f]-mu)/sd)@fit_cox((X[tr]-mu)/sd,t[tr],e[tr]),e[f])
            if np.isnan(o): continue
            bl={"Static (McCabe)":oriented(t[f],S.cc_lm.values[f],e[f]),
                "Static (Halstead? via mi)":oriented(t[f],S.mi_lm.values[f],e[f]),
                "Size (LOC)":oriented(t[f],S.loc_lm.values[f],e[f]),
                "Coupling (fan_in)":oriented(t[f],S.fan_in_lm.values[f],e[f]),
                "Process (n_authors)":oriented(t[f],S.n_authors.values[f],e[f])}
            for k,v in bl.items(): out[k].append(v)
            out["Cox trajectory (ours)"].append(o); dif.append(o-max(bl.values()))
    return {k:np.array(v) for k,v in out.items()}, np.array(dif)

res,dif=repeated_cv()
rows=[]
for k in sorted(res,key=lambda x:-res[x].mean()):
    v=res[k];lo,hi=np.percentile(v,[2.5,97.5])
    rows.append(dict(model=k,concordance=round(v.mean(),3),ci_low=round(lo,3),ci_high=round(hi,3)))
T2=pd.DataFrame(rows); T2.to_csv("table2_results.csv",index=False)
print("TABLE 2 (repeated 10x5 CV):"); display(T2)
lo,hi=np.percentile(dif,[2.5,97.5])
print(f"Paired gain over best baseline: {dif.mean():+.3f}  95% CI [{lo:+.3f},{hi:+.3f}]  "
      f"-> {'SUPPORTED' if lo>0 else 'NOT SUPPORTED'}")

In [ ]:
# 8) TABLE 3 -- Leave-One-Repo-Out (generalization to an UNSEEN project) -- the hard test
rows=[]
for repo in sorted(S.repo.unique()):
    te=(S.repo==repo).values; tr=~te
    if e[te].sum()<5: continue
    muL,sdL=X[tr].mean(0),X[tr].std(0)+1e-9
    b=fit_cox((X[tr]-muL)/sdL,t[tr],e[tr])
    ours=concordance(t[te],((X[te]-muL)/sdL)@b,e[te])
    mcc=oriented(t[te],S.cc_lm.values[te],e[te])
    rows.append(dict(held_out=repo,files=int(te.sum()),events=int(e[te].sum()),
                     ours=round(ours,3),mccabe=round(mcc,3)))
T3=pd.DataFrame(rows); T3.to_csv("table3_leave_one_repo_out.csv",index=False)
print("TABLE 3 (Leave-One-Repo-Out):"); display(T3)
print(f"mean ours={T3.ours.mean():.3f} | mean McCabe={T3.mccabe.mean():.3f} | "
      f"ours wins {int((T3.ours>T3.mccabe).sum())}/{len(T3)}")
print("NOTE: this is far below the pooled-CV number. Report BOTH; the gap is the generalization story.")

In [ ]:
# 9) TABLE 4 -- sensitivity (L x DELTA), cross-validated, does the gain survive?
rows=[]
for L in [2,3,5]:
    for d in [1.0,2.0,3.0]:
        Sx=build_landmark(df,L=L,delta=d)
        if len(Sx)<150 or Sx.event.sum()<40: continue
        tt=Sx.duration_days.values.astype(float); ee=Sx.event.values.astype(int)
        XX=Sx[COV].values; m2,s2=XX.mean(0),XX.std(0)+1e-9
        r=np.random.default_rng(1); dd=[]
        for _ in range(4):
            idx=r.permutation(len(Sx))
            for f in np.array_split(idx,5):
                tr=np.setdiff1d(idx,f)
                if ee[tr].sum()<5 or ee[f].sum()<2: continue
                o=concordance(tt[f],((XX[f]-m2)/s2)@fit_cox((XX[tr]-m2)/s2,tt[tr],ee[tr]),ee[f])
                bb=max(oriented(tt[f],Sx.cc_lm.values[f],ee[f]),
                       oriented(tt[f],Sx.n_authors.values[f],ee[f]),
                       oriented(tt[f],Sx.loc_lm.values[f],ee[f]))
                if not np.isnan(o): dd.append(o-bb)
        dd=np.array(dd); lo,hi=np.percentile(dd,[2.5,97.5])
        rows.append(dict(L=L,delta=d,files=len(Sx),events=int(Sx.event.sum()),
            diff=round(dd.mean(),3),ci_low=round(lo,3),ci_high=round(hi,3),
            significant=("yes" if lo>0 else "NO")))
T4=pd.DataFrame(rows); T4.to_csv("table4_sensitivity.csv",index=False)
print("TABLE 4 (sensitivity):"); display(T4)
print(f"significant in {int((T4.significant=='yes').sum())}/{len(T4)} configurations")

In [ ]:
# 10) KM figure + artifacts
def km(d,e):
    d=np.asarray(d,float);e=np.asarray(e,int);T=[];Sv=[];s=1.
    for tt in np.unique(d):
        atr=int((d>=tt).sum());dth=int(e[d==tt].sum())
        if atr>0: s*=(1-dth/atr)
        T.append(tt);Sv.append(s)
    return np.array(T),np.array(Sv)
plt.figure(figsize=(9,5))
q=S.n_authors.median()
for lab,msk,c in [("All",np.ones(len(S),bool),"black"),
                  (f"n_authors<={q:.0f}",(S.n_authors<=q).values,"#2c7fb8"),
                  (f"n_authors>{q:.0f}",(S.n_authors>q).values,"#d95f0e")]:
    sub=S[msk]
    if len(sub)>20:
        T,Sv=km(sub.duration_days,sub.event)
        plt.step(T,Sv,where="post",lw=2,color=c,label=f"{lab} (n={len(sub)})")
plt.xlabel(f"Days since landmark (first {LANDMARK_L} commits)")
plt.ylabel(f"P(max_cc not yet risen by {DELTA})")
plt.title("Complexity-debt accumulation"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig("figure1_km.png",dpi=150); plt.show()

for f in ["complexity_long_final.csv","survival_landmark_final.csv","table2_results.csv",
          "table3_leave_one_repo_out.csv","table4_sensitivity.csv","figure1_km.png"]:
    if os.path.exists(f): print(" -",f,f"({os.path.getsize(f)} B)")

## Paper skeleton (what the tables support)

**Title direction:** *Early development dynamics, not initial code structure, predict complexity accumulation in Python repositories: a survival analysis.*

**Defensible claims:**
- **RQ1 — What predicts accumulation?** Initial static complexity barely beats random (McCabe ≈ 0.54); early process dynamics (development tempo `lm_days`, code growth, author count) carry the signal. *Table 2.*
- **RQ2 — Does modelling the trajectory help?** Yes, under pooled CV: +0.085 [+0.035, +0.152] over the best baseline, significant in 8/9 configurations. *Tables 2, 4.*
- **RQ3 — Does it generalize to unseen projects?** **Only weakly** — LORO mean 0.599 vs McCabe 0.588, losing on 4/14 repos. This is a limitation, stated openly. *Table 3.*

**Negative results to report, not bury:**
- Structural/coupling covariates add nothing over process covariates (+0.014 [−0.026, +0.050]).
- A nonlinear model adds nothing (+0.021 [−0.017, +0.078]).
- AI-authorship is unobservable from commit trailers (0/20,014). No AI covariate.

**Do NOT interpret coefficient signs** — `fan_out`/`n_imports` collinearity (r=0.79) inverts them. The predictive tables are the result; the coefficient plot is diagnostic only.

**Venue:** the modest effect and honest negatives fit an empirical-study slot at **IST** or **EMSE** better than a "predictor" claim at TOSEM. The generalization gap (Table 3) is the reviewer's first question — it is already answered.

**Threats to validity:** commit cap truncates late life (inflates censoring); fan-in under-counts (mined commits only); module resolution collides same-named modules; renames restart trajectories; mature-OSS population is complexity-inert (most files never grow); single-language (Python).

In [ ]:
# 11) Package ALL generated artifacts into ./moatastm and download as a zip
import os, glob, shutil

OUT = "moatastm"
if os.path.exists(OUT): shutil.rmtree(OUT)
os.makedirs(OUT, exist_ok=True)

# only the artifacts THIS study produces (data tables + figures)
EXPECTED = [
    "complexity_long_final.csv", "survival_landmark_final.csv",
    "table2_results.csv", "table3_leave_one_repo_out.csv",
    "table4_sensitivity.csv", "figure1_km.png",
]
# plus any other csv/png created this run, excluding config/system files
EXCLUDE = {"rclone-mount-config.json"}
extra = [f for f in glob.glob("*.csv") + glob.glob("*.png")
         if f not in EXPECTED and f not in EXCLUDE and not f.startswith(".")]

collected = []
for f in EXPECTED + extra:
    if os.path.isfile(f) and os.path.getsize(f) > 0:
        shutil.copy(f, os.path.join(OUT, os.path.basename(f)))
        collected.append(f)

print(f"packaged {len(collected)} files into ./{OUT}/:")
for f in sorted(collected): print("   -", f, f"({os.path.getsize(f)} B)")

zip_path = shutil.make_archive(OUT, "zip", OUT)
print(f"\ncreated {zip_path} ({os.path.getsize(zip_path)} B)")

try:
    from google.colab import files
    files.download(zip_path)
    print("download started (check your browser).")
except Exception:
    print("Not on Colab -> find the moatastm/ folder and moatastm.zip in the Files panel.")